# ESA GeoFM Challenge — v5  (the one that actually trains)

Same fusion architecture as v4 (SE-UNet + ASPP + TM/Thor cross-attention + dual height heads),
**but v4 never learned**: every batch's loss was non-finite, so the NaN-skip guard dropped all of
them (`trn=0.000`, `val=nan`, `score=0`, `RMSE=nan`) and the weights stayed at init.

**Root cause.** PREP normalises embeddings and clips to `±60000` as fp16. Near-constant
embedding dims (std≈0) blow up to ±60000 spikes; under `autocast` the first conv overflows fp16
(max≈65504) → Inf → BatchNorm → NaN → NaN loss on *every* batch.

**v5 fixes + extends:**
- `FEAT_CLAMP` clamps normalised features to ±15 at load **and** at inference (works on existing zips — no re-PREP).
- A **diagnostic cell** (§D2) runs one batch and proves the loss is finite before you spend GPU time.
- NaN-safe height loss guard.
- Light **embedding-space augmentation** (noise + channel-drop) to close the water val→test gap.
- Multi-seed ensemble + multi-scale TTA already wired (just change `SEED` and re-run).

**Target proxy 0.50** (v4-line leaderboard was 0.4112 with the weights stuck at init equivalent /
partial). Train ≥2 seeds, then run the ensemble cell.


## A — PREP (run once — skips existing zips automatically)

In [ ]:
# Set RUN_PREP=True the first time (or to rebuild zips).
# All existing zips are skipped automatically — only missing ones are built.
RUN_PREP = False

if RUN_PREP:
    from google.colab import drive as _drive
    _drive.mount('/content/drive', force_remount=False)
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q','rasterio'], check=True)

    import re, time, io, zipfile, shutil, threading, json
    from pathlib import Path
    import numpy as np, torch, torch.nn.functional as F, rasterio, warnings
    from rasterio.io import MemoryFile
    from concurrent.futures import ThreadPoolExecutor
    warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

    ROOT   = Path('/content/drive/MyDrive/ESA_Challenge')
    OUT_TR = ROOT/'kaggle_transfer_v5'/'train'; OUT_TR.mkdir(parents=True, exist_ok=True)
    OUT_TE = ROOT/'kaggle_transfer_v5'/'test';  OUT_TE.mkdir(parents=True, exist_ok=True)
    IO_WORKERS = 16
    _lock = threading.Lock()

    def patch_id(stem):
        m = re.search(r'_(\d{4})_', stem); return m.group(1) if m else None

    def safe_load_tif(path, target_hw=None, retries=6):
        last = None
        for a in range(retries):
            try:
                with open(path,'rb') as fh: data = fh.read()
                with MemoryFile(data) as mf, mf.open() as src:
                    arr = src.read().astype(np.float32)
                if target_hw and arr.shape[-2:] != (target_hw, target_hw):
                    arr = F.interpolate(torch.from_numpy(arr)[None],
                                        size=(target_hw,target_hw),
                                        mode='bilinear',align_corners=False)[0].numpy()
                return arr
            except Exception as e:
                last = e
                if 'not connected' in str(e).lower() or getattr(e,'errno',None)==107:
                    with _lock:
                        try: _drive.mount('/content/drive', force_remount=True)
                        except Exception: pass
                time.sleep(1.0*(a+1))
        raise RuntimeError(f'read {Path(path).name}: {last}')

    stats = np.load(ROOT/'norm_stats.npy', allow_pickle=True).item()
    def mstd(key):
        return (stats[key]['mean'].reshape(-1,1,1).astype(np.float32),
                stats[key]['std'].reshape(-1,1,1).astype(np.float32))

    def compute_stats(files, label):
        samp = files[::max(1,len(files)//300)]
        print(f'Computing {label} stats from {len(samp)} samples…')
        s1=s2=cnt=None
        for a in ThreadPoolExecutor(IO_WORKERS).map(lambda p: safe_load_tif(p), samp):
            a=a.reshape(a.shape[0],-1); m=~np.isnan(a); a0=np.where(m,a,0.0)
            if s1 is None: s1,s2,cnt=a0.sum(1),(a0**2).sum(1),m.sum(1)
            else: s1+=a0.sum(1); s2+=(a0**2).sum(1); cnt+=m.sum(1)
        mean=s1/np.maximum(cnt,1); var=np.maximum(s2/np.maximum(cnt,1)-mean**2,1e-6)
        return mean.reshape(-1,1,1).astype(np.float32), np.sqrt(var).reshape(-1,1,1).astype(np.float32)

    AE_MEAN, AE_STD = mstd('alphaearth_emb')
    TM_MEAN, TM_STD = mstd('terramind_s2_emb')

    def find_dir(parent, names):
        for n in names:
            d = parent/n
            if d.is_dir() and any(d.glob('*.tif')): return d
        return None

    # Tessera
    TESS_TR = find_dir(ROOT/'train', ['tessera_emb','tessera','tessera_s2_emb'])
    TESS_TE = find_dir(ROOT/'test',  ['tessera_emb','tessera','tessera_test_emb','tessera_test_s2_emb'])
    tkey = next((k for k in stats if 'tess' in k.lower()), None)
    if tkey:
        TS_MEAN, TS_STD = mstd(tkey)
    else:
        TS_MEAN, TS_STD = compute_stats(sorted(TESS_TR.glob('*.tif')), 'Tessera')
        stats['tessera_emb'] = {'mean': TS_MEAN.squeeze().astype(np.float32),
                                'std':  TS_STD.squeeze().astype(np.float32)}

    # Thor
    THOR_TR = find_dir(ROOT/'train', ['thor_s2_emb','thor_emb','thor'])
    THOR_TE = find_dir(ROOT/'test',  ['thor_s2_emb','thor_emb','thor'])
    assert THOR_TR is not None, 'Cannot find Thor S2 train dir under ESA_Challenge/train/'
    tkey2 = next((k for k in stats if 'thor' in k.lower()), None)
    if tkey2:
        TH_MEAN, TH_STD = mstd(tkey2)
    else:
        TH_MEAN, TH_STD = compute_stats(sorted(THOR_TR.glob('*.tif')), 'Thor')
        stats['thor_s2_emb'] = {'mean': TH_MEAN.squeeze().astype(np.float32),
                                'std':  TH_STD.squeeze().astype(np.float32)}

    np.save(ROOT/'norm_stats.npy', stats)
    shutil.copy2(ROOT/'norm_stats.npy', OUT_TR/'norm_stats.npy')

    def prenorm(arr, mean, std):
        x = np.nan_to_num((arr-mean)/(std+1e-6), nan=0.0)
        return np.clip(x, -60000, 60000).astype(np.float16)

    def build_zip(src_dir, out_dir, zname, mode, mean=None, std=None, target_hw=256):
        final = out_dir/zname
        if final.exists(): print(f'{zname}: exists, skip'); return
        local = Path('/content')/zname
        tifs  = sorted(src_dir.glob('*.tif'))
        print(f'{zname}: packing {len(tifs)} files…')
        def work(p):
            a = safe_load_tif(p, target_hw=target_hw)
            arr = (np.nan_to_num(a, nan=0.0).astype(np.float16) if mode=='label'
                   else prenorm(a, mean, std))
            bio = io.BytesIO(); np.save(bio, arr); return (p.stem+'.npy', bio.getvalue())
        t0 = time.time()
        with zipfile.ZipFile(local,'w',zipfile.ZIP_STORED) as zf, ThreadPoolExecutor(IO_WORKERS) as ex:
            for i,(name,data) in enumerate(ex.map(work, tifs)):
                zf.writestr(name, data)
                if (i+1)%250==0: print(f'  {i+1}/{len(tifs)}  {(i+1)/(time.time()-t0):.1f}/s')
        print(f'  {local.stat().st_size/1e9:.1f} GB → Drive…')
        shutil.copy2(local, final); local.unlink()
        print(f'  done → {final}')

    # ── TRAIN ──
    build_zip(ROOT/'train'/'alphaearth_emb', OUT_TR,'ae_train_npy.zip','prenorm',AE_MEAN,AE_STD)
    build_zip(TESS_TR,                       OUT_TR,'ts_train_npy.zip','prenorm',TS_MEAN,TS_STD)
    build_zip(ROOT/'train'/'terramind_s2_emb',OUT_TR,'tm_train_npy.zip','prenorm',TM_MEAN,TM_STD,target_hw=None)
    build_zip(THOR_TR,                        OUT_TR,'th_train_npy.zip','prenorm',TH_MEAN,TH_STD,target_hw=None)
    build_zip(ROOT/'train'/'labels',          OUT_TR,'lbl_train_npy.zip','label')
    # ── TEST ──
    build_zip(ROOT/'test'/'alphaearth_emb',   OUT_TE,'ae_test_npy.zip','prenorm',AE_MEAN,AE_STD)
    build_zip(TESS_TE,                         OUT_TE,'ts_test_npy.zip','prenorm',TS_MEAN,TS_STD)
    build_zip(ROOT/'test'/'terramind_test_s2_emb',OUT_TE,'tm_test_npy.zip','prenorm',TM_MEAN,TM_STD,target_hw=None)
    if THOR_TE: build_zip(THOR_TE, OUT_TE,'th_test_npy.zip','prenorm',TH_MEAN,TH_STD,target_hw=None)
    # submission name map
    names = {}
    for p in sorted((ROOT/'test'/'terramind_test_s2_emb').glob('*.tif')):
        pid = patch_id(p.stem)
        if pid: names[pid] = re.sub(r'^s2_','',re.sub(r'_embeddings$','',p.stem))
    (OUT_TE/'test_names.json').write_text(json.dumps(names))
    print(f'\nPREP DONE — train: {OUT_TR}\n           test : {OUT_TE}')
else:
    print('PREP skipped — set RUN_PREP=True to rebuild zips.')


PREP skipped — set RUN_PREP=True to rebuild zips.


## B — Config  *(edit SEED here, nowhere else)*

In [ ]:
from google.colab import drive as _drive
_drive.mount('/content/drive', force_remount=False)

import re, random, warnings, shutil, zipfile, json, time, io, os
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Modalities ────────────────────────────────────────────────────────────────
USE_TESSERA = True   # AlphaEarth (64ch) + Tessera (128ch) → 192ch spatial branch
USE_THOR    = True   # TerraMind (768ch) + Thor (768ch)   → 1536ch token branch

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/ESA_Challenge')
DATA_TRAIN = DRIVE_ROOT/'kaggle_transfer_v5'/'train'
DATA_TEST  = DRIVE_ROOT/'kaggle_transfer_v5'/'test'
TMP        = Path('/content/tmp'); TMP.mkdir(parents=True, exist_ok=True)

# ── Training ──────────────────────────────────────────────────────────────────
SEED       = 42      # ← change to 123 / 2024 for extra ensemble seeds
BATCH_SIZE = 16
EPOCHS     = 80
LR         = 3e-4
VAL_FRAC   = 0.15
EMA_DECAY  = 0.999
NUM_WORKERS= 2       # set 0 if workers crash
TIME_BUDGET_H = 11.0

# ── v5 stability / augmentation ───────────────────────────────────────────────
FEAT_CLAMP  = 15.0   # clamp normalised embeddings to ±this. PREP clips to ±60000; near-constant
                     # embedding dims (std≈0) become ±60000 spikes that overflow fp16 under autocast
                     # → Inf → NaN loss on every batch. ±15 keeps all real signal, kills the spikes.
FEAT_NOISE  = 0.03   # std of gaussian noise added to embeddings while training (0 = off)
CHAN_DROP_P = 0.05   # fraction of spatial feature channels randomly zeroed while training (0 = off)

# ── Checkpoint dir (one per seed — never collides with v2) ────────────────────
WORK = DRIVE_ROOT/f'colab_work_v5_s{SEED}'
WORK.mkdir(parents=True, exist_ok=True)
CKPT_BEST = WORK/'final_best.pth'
CKPT_LAST = WORK/'final_last.pth'
HISTORY_F = WORK/'final_history.json'
THRESH_F  = WORK/'final_thresholds.json'
SUB_ZIP   = WORK/'submission.zip'

# ── Misc ──────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
T_START = time.time()
def hours_left(): return TIME_BUDGET_H - (time.time()-T_START)/3600
def disk_used_gb(p='/content'):
    u = shutil.disk_usage(p); return u.used/1e9, u.free/1e9

print(f'Device : {DEVICE}')
if DEVICE.type=='cuda': print(f'GPU    : {torch.cuda.get_device_name(0)}')
u,fr = disk_used_gb(); print(f'SSD    : {u:.0f} GB used / {fr:.0f} GB free')
print(f'WORK   : {WORK}')
assert fr > 60, f'Only {fr:.0f} GB free — need ~60 GB. Connect a fresh runtime.'


Mounted at /content/drive
Device : cuda
GPU    : NVIDIA L4
SSD    : 51 GB used / 202 GB free
WORK   : /content/drive/MyDrive/ESA_Challenge/colab_work_v5_s42


## C — Stage zips to local SSD & zip reader

In [ ]:
def stage_local(src_dir, dest, files, retries=6):
    dest.mkdir(parents=True, exist_ok=True)
    for f in files:
        d = dest/f
        for a in range(retries):
            try:
                s = src_dir/f
                if not s.exists(): raise OSError(f'not visible: {s}')
                ssz = s.stat().st_size
                if d.exists() and d.stat().st_size==ssz:
                    print(f'  present: {f}'); break
                t0 = time.time()
                print(f'  copying {f} ({ssz/1e9:.1f} GB), try {a+1}/{retries}…', flush=True)
                shutil.copy2(s, d)
                if d.stat().st_size != ssz: raise OSError('size mismatch (Drive dropped)')
                print(f'    done in {time.time()-t0:.0f}s'); break
            except OSError as e:
                print(f'    ! {e}  → remounting & retrying')
                try:
                    if d.exists(): d.unlink()
                except Exception: pass
                try: _drive.mount('/content/drive', force_remount=True)
                except Exception as me: print('      remount:', me)
                time.sleep(3*(a+1))
        else:
            raise RuntimeError(f'could not stage {f} after {retries} tries')

TRAIN_ZIPS = (['ae_train_npy.zip','tm_train_npy.zip','lbl_train_npy.zip']
              + (['ts_train_npy.zip'] if USE_TESSERA else [])
              + (['th_train_npy.zip'] if USE_THOR    else []))
stage_local(DATA_TRAIN, TMP, TRAIN_ZIPS + ['norm_stats.npy'])
for z in TRAIN_ZIPS:
    assert (TMP/z).exists(), f'missing {z}'
u,fr = disk_used_gb(); print(f'After staging: {u:.0f} GB used / {fr:.0f} GB free')

def patch_id(stem):
    m = re.search(r'(?<!\d)(\d{4})(?!\d)', stem); return m.group(1) if m else None

class ZipNpyReader:
    def __init__(self, zip_path):
        self.zip_path = str(zip_path); self._zf = None
        with zipfile.ZipFile(self.zip_path) as zf:
            self.names = {patch_id(Path(n).stem): n for n in zf.namelist()
                          if n.endswith('.npy') and patch_id(Path(n).stem)}
    def ids(self): return set(self.names)
    def get(self, pid, _retries=3):
        name = self.names[pid]
        for a in range(_retries):
            try:
                if self._zf is None: self._zf = zipfile.ZipFile(self.zip_path)
                with self._zf.open(name) as f:
                    return np.load(io.BytesIO(f.read()))
            except (zipfile.BadZipFile, OSError):
                try: self._zf.close()
                except Exception: pass
                self._zf = None
                if a == _retries-1: raise

print('Stage & readers ready.')


  copying ae_train_npy.zip (17.0 GB), try 1/6…
    done in 264s
  copying tm_train_npy.zip (0.8 GB), try 1/6…
    done in 15s
  copying lbl_train_npy.zip (1.1 GB), try 1/6…
    done in 21s
  copying ts_train_npy.zip (34.0 GB), try 1/6…
    done in 662s
  copying th_train_npy.zip (0.8 GB), try 1/6…
    done in 22s
  copying norm_stats.npy (0.0 GB), try 1/6…
    done in 1s
After staging: 105 GB used / 148 GB free
Stage & readers ready.


## D — Dataset & loaders

In [ ]:
ae_reader = ZipNpyReader(TMP/'ae_train_npy.zip')
ts_reader = ZipNpyReader(TMP/'ts_train_npy.zip') if USE_TESSERA else None
th_reader = ZipNpyReader(TMP/'th_train_npy.zip') if USE_THOR    else None

# TerraMind + labels loaded entirely into RAM (small enough at fp16)
TM_CACHE, TH_CACHE, SEG_CACHE, HGT_CACHE = {}, {}, {}, {}
_tmz = ZipNpyReader(TMP/'tm_train_npy.zip')
for pid in _tmz.ids(): TM_CACHE[pid] = _tmz.get(pid).astype(np.float16)
if USE_THOR:
    _thz = ZipNpyReader(TMP/'th_train_npy.zip')
    for pid in _thz.ids(): TH_CACHE[pid] = _thz.get(pid).astype(np.float16)
_lblz = ZipNpyReader(TMP/'lbl_train_npy.zip')
for pid in _lblz.ids():
    lbl = _lblz.get(pid)
    SEG_CACHE[pid] = np.nan_to_num(lbl[:3], nan=0.0).clip(0,1).astype(np.float16)
    HGT_CACHE[pid] = np.log1p(np.nan_to_num(lbl[3:4], nan=0.0).clip(0,None)).astype(np.float16)

class FusionDataset(Dataset):
    def __init__(self, ids, ae_r, ts_r=None, th_r=None, augment=False):
        self.ids, self.ae_r, self.ts_r, self.th_r, self.augment = ids,ae_r,ts_r,th_r,augment
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        pid = self.ids[idx]
        parts = [self.ae_r.get(pid).astype(np.float32)]
        if self.ts_r: parts.append(self.ts_r.get(pid).astype(np.float32))
        spatial = torch.from_numpy(np.concatenate(parts, axis=0))
        tm  = torch.from_numpy(TM_CACHE[pid].astype(np.float32))
        th  = torch.from_numpy(TH_CACHE[pid].astype(np.float32)) if USE_THOR else torch.zeros(768,16,16)
        seg = torch.from_numpy(SEG_CACHE[pid].astype(np.float32))
        h   = torch.from_numpy(HGT_CACHE[pid].astype(np.float32))
        # ── v5: kill fp16-overflow spikes baked into the zips (±60000 → NaN loss) ──
        spatial = torch.nan_to_num(spatial).clamp_(-FEAT_CLAMP, FEAT_CLAMP)
        tm      = torch.nan_to_num(tm).clamp_(-FEAT_CLAMP, FEAT_CLAMP)
        th      = torch.nan_to_num(th).clamp_(-FEAT_CLAMP, FEAT_CLAMP)
        if self.augment:
            # geometric (applied to every tensor so labels stay aligned)
            tensors = (spatial, tm, th, seg, h)
            if random.random()>0.5: tensors = tuple(torch.flip(t,[-1]) for t in tensors)
            if random.random()>0.5: tensors = tuple(torch.flip(t,[-2]) for t in tensors)
            k = random.randint(0,3)
            if k: tensors = tuple(torch.rot90(t,k,[-2,-1]) for t in tensors)
            spatial, tm, th, seg, h = tensors
            # feature-space (embeddings only — regularises the water val→test gap)
            if FEAT_NOISE > 0:
                spatial = spatial + torch.randn_like(spatial)*FEAT_NOISE
                tm      = tm      + torch.randn_like(tm)*FEAT_NOISE
                th      = th      + torch.randn_like(th)*FEAT_NOISE
            if CHAN_DROP_P > 0:
                keep    = (torch.rand(spatial.shape[0],1,1) > CHAN_DROP_P).float()
                spatial = spatial * keep
            spatial = spatial.clamp_(-FEAT_CLAMP, FEAT_CLAMP)
            tm      = tm.clamp_(-FEAT_CLAMP, FEAT_CLAMP)
            th      = th.clamp_(-FEAT_CLAMP, FEAT_CLAMP)
        return spatial, tm, th, seg, h

# Intersect available patch IDs across all modalities
key_sets = [ae_reader.ids(), set(TM_CACHE), set(SEG_CACHE), set(HGT_CACHE)]
if USE_TESSERA: key_sets.append(ts_reader.ids())
if USE_THOR:    key_sets.append(set(TH_CACHE))
ids = sorted(set.intersection(*key_sets))
rng = random.Random(SEED); rng.shuffle(ids)
n_val = int(len(ids)*VAL_FRAC)
val_ids, trn_ids = ids[:n_val], ids[n_val:]

SPATIAL_CH = 64 + (ts_reader.get(ids[0]).shape[0] if USE_TESSERA else 0)
TM_CH      = TM_CACHE[ids[0]].shape[0]   # 768
THOR_CH    = 768 if USE_THOR else 0
print(f'Patches  : {len(ids)}  train={len(trn_ids)}  val={len(val_ids)}')
print(f'Spatial  : {SPATIAL_CH}ch  |  TM: {TM_CH}ch  Thor: {THOR_CH}ch')

# Class-balanced sampler: 3× weight for patches with buildings
build_frac   = np.array([SEG_CACHE[p][0].mean() for p in trn_ids], dtype=np.float32)
build_weight = torch.from_numpy(np.where(build_frac > 0.05, 3.0, 1.0).astype(np.float32))
trn_sampler  = torch.utils.data.WeightedRandomSampler(build_weight, len(trn_ids), replacement=True)

trn_ds = FusionDataset(trn_ids, ae_reader, ts_reader, th_reader, augment=True)
val_ds = FusionDataset(val_ids, ae_reader, ts_reader, th_reader, augment=False)
trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, sampler=trn_sampler,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0)

t0 = time.time()
for i,_ in zip(range(3), trn_loader): pass
print(f'Loader warmup: {(time.time()-t0)/3*1000:.0f} ms/batch')


Patches  : 2024  train=1721  val=303
Spatial  : 192ch  |  TM: 768ch  Thor: 768ch
Loader warmup: 8179 ms/batch


## E — Model: SE-UNet + ASPP + Dual Height Heads

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__(); h = max(ch//r, 4)
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(ch,h,1), nn.ReLU(inplace=True),
                                nn.Conv2d(h,ch,1), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x)

class DoubleConv(nn.Module):
    def __init__(self, ic, oc, se=True):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(ic,oc,3,padding=1,bias=False), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
            nn.Conv2d(oc,oc,3,padding=1,bias=False), nn.BatchNorm2d(oc), nn.ReLU(inplace=True))
        self.se = SEBlock(oc) if se else nn.Identity()
    def forward(self, x): return self.se(self.b(x))

class ASPP(nn.Module):
    # Multi-scale context at bottleneck. Rates 1/6/12/18 cover 8-144m at 10m/px.
    def __init__(self, ch=256, rates=(1,6,12,18)):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(ch,ch,3,padding=r,dilation=r,bias=False),
                          nn.BatchNorm2d(ch), nn.ReLU(inplace=True))
            for r in rates])
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(ch,ch,1), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(nn.Conv2d(ch*(len(rates)+1),ch,1,bias=False),
                                  nn.BatchNorm2d(ch), nn.ReLU(inplace=True))
    def forward(self, x):
        outs = [b(x) for b in self.branches]
        outs.append(F.interpolate(self.pool(x), size=x.shape[-2:], mode='bilinear', align_corners=False))
        return self.fuse(torch.cat(outs, dim=1))

class CrossAttn(nn.Module):
    # Bottleneck spatial queries attend over TM+Thor token set.
    def __init__(self, spatial_ch=256, token_ch=1536, attn_dim=256, n_heads=4):
        super().__init__()
        self.q_proj  = nn.Conv2d(spatial_ch, attn_dim, 1)
        self.kv_proj = nn.Linear(token_ch, attn_dim)
        self.attn    = nn.MultiheadAttention(attn_dim, n_heads, batch_first=True, dropout=0.1)
        self.norm    = nn.LayerNorm(attn_dim)
        self.out     = nn.Conv2d(attn_dim, spatial_ch, 1)
    def forward(self, x, tokens):
        # x: (B,C,H,W)  tokens: (B, token_ch, 16, 16)
        B,C,H,W = x.shape
        KV = self.kv_proj(tokens.flatten(2).permute(0,2,1))   # (B,256,attn_dim)
        Q  = self.q_proj(x).flatten(2).permute(0,2,1)         # (B,H*W,attn_dim)
        o,_ = self.attn(Q, KV, KV)
        o = self.norm(o).permute(0,2,1).reshape(B,C,H,W)
        return x + self.out(o)

class UNetDec(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__(); self.conv = DoubleConv(in_ch+skip_ch, out_ch)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class HeightBranch(nn.Module):
    # Single-class height decoder: features + class seg logit -> n_bins distribution.
    def __init__(self, in_ch=64, n_bins=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch+1, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128,      64, 3, padding=1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64, n_bins, 1))
    def forward(self, feat, seg_logit_1ch):
        return self.net(torch.cat([feat, seg_logit_1ch], dim=1))   # (B, n_bins, H, W)

HEIGHT_BINS = 64; HMAX_LOG = 5.5
BIN_CENTERS = torch.linspace(0.0, HMAX_LOG, HEIGHT_BINS, device=DEVICE)

class FusionNetV4(nn.Module):
    # SE-UNet + ASPP + CrossAttn + dual height heads.
    def __init__(self, spatial_ch, tm_ch=768, thor_ch=768, n_bins=HEIGHT_BINS):
        super().__init__()
        token_ch = tm_ch + thor_ch   # 1536 (or 768 if thor disabled)
        self.enc1 = DoubleConv(spatial_ch, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.bot  = DoubleConv(256, 256)
        self.pool = nn.MaxPool2d(2)
        # Token projection + residual injection
        self.tok_proj  = nn.Sequential(nn.Upsample(size=(32,32), mode='bilinear', align_corners=False),
                                       nn.Conv2d(token_ch, 256, 1))
        self.aspp      = ASPP(256)
        self.cross_attn= CrossAttn(256, token_ch, attn_dim=256, n_heads=4)
        self.dec3 = UNetDec(256, 256, 256)
        self.dec2 = UNetDec(256, 128, 128)
        self.dec1 = UNetDec(128,  64,  64)
        self.seg_head   = nn.Conv2d(64, 3, 1)
        self.h_build    = HeightBranch(64, n_bins)
        self.h_veg      = HeightBranch(64, n_bins)

    def forward(self, x, tm, th=None):
        # Spatial encoder
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        b  = self.bot(self.pool(s3))
        # Token fusion
        tokens = torch.cat([tm, th], dim=1) if th is not None else tm   # (B, 1536, 16, 16)
        b = self.aspp(b)
        b = b + self.tok_proj(tokens)          # residual token injection
        b = self.cross_attn(b, tokens)         # cross-attention refinement
        # Decoder
        d3 = self.dec3(b,  s3)
        d2 = self.dec2(d3, s2)
        d1 = self.dec1(d2, s1)
        seg = self.seg_head(d1)                # (B, 3, 256, 256) logits
        # Dual height heads
        bc = BIN_CENTERS.view(1,-1,1,1)
        hbl = self.h_build(d1, seg[:,0:1])     # (B, K, 256, 256)
        hvl = self.h_veg  (d1, seg[:,1:2])
        h_b = (torch.softmax(hbl.float(), 1) * bc).sum(1, keepdim=True)
        h_v = (torch.softmax(hvl.float(), 1) * bc).sum(1, keepdim=True)
        # Route: build pixels → h_b, veg pixels → h_v, blend elsewhere
        pb = torch.sigmoid(seg[:,0:1]); pv = torch.sigmoid(seg[:,1:2])
        h_out = torch.where(pb >= pv, h_b, h_v)
        return seg, h_out, hbl, hvl, h_b, h_v

model = FusionNetV4(SPATIAL_CH, TM_CH, THOR_CH).to(DEVICE).to(memory_format=torch.channels_last)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')
with torch.no_grad():
    _x  = torch.randn(2, SPATIAL_CH, 256, 256, device=DEVICE).to(memory_format=torch.channels_last)
    _tm = torch.randn(2, TM_CH,  16, 16, device=DEVICE)
    _th = torch.randn(2, THOR_CH,16, 16, device=DEVICE) if USE_THOR else None
    _seg, _h, _hbl, _hvl, _hb, _hv = model(_x, _tm, _th)
print(f'Smoke test — seg:{tuple(_seg.shape)}  h:{tuple(_h.shape)}  bins:{tuple(_hbl.shape)}')
torch.cuda.empty_cache()


Parameters: 9,219,339
Smoke test — seg:(2, 3, 256, 256)  h:(2, 1, 256, 256)  bins:(2, 64, 256, 256)


## F — Loss, metrics & proxy score

In [ ]:
# ── Segmentation ──────────────────────────────────────────────────────────────
def focal_bce(logits, target, gamma=2.0, w=1.0):
    logits = logits.float(); p = torch.sigmoid(logits)
    ce  = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    pt  = target*p + (1-target)*(1-p)
    return (w * (1-pt).clamp(min=1e-6)**gamma * ce).mean()

def soft_iou_loss(logits, target, ch, eps=1e-6):
    p = torch.sigmoid(logits[:,ch].float()); t = target[:,ch]
    return 1 - (p*t).sum() / ((p+t-p*t).sum() + eps)

def gradient_loss(pred, target):
    def g(x): return x[:,:,:,1:]-x[:,:,:,:-1], x[:,:,1:,:]-x[:,:,:-1,:]
    a,b = g(pred); c,d = g(target)
    return F.l1_loss(a,c) + F.l1_loss(b,d)

# ── Height (per-branch) ───────────────────────────────────────────────────────
def branch_height_loss(h_pred, h_logits, h_t, seg_prob):
    # h_pred: (B,1,H,W) soft-argmax height, h_logits: (B,K,H,W) bins, seg_prob: coverage weight
    h_pred = h_pred.float(); h_t = h_t.float(); seg_prob = seg_prob.float().clamp(0,1)
    # Weight: height-present × coverage-confidence
    w = (1.0 + 2.0*(h_t>0).float()) * (0.5 + seg_prob)
    huber = (w * F.huber_loss(h_pred, h_t, delta=1.0, reduction='none')).mean()
    with torch.no_grad():
        idx = torch.bucketize(h_t.squeeze(1).contiguous(),
                              BIN_CENTERS).clamp(0, HEIGHT_BINS-1)
    h_logits = torch.nan_to_num(h_logits.float(), nan=0.0, posinf=30.0, neginf=-30.0)
    ce = (w * F.cross_entropy(h_logits, idx, reduction='none').unsqueeze(1)).mean()
    return huber + 0.5*ce + 0.1*gradient_loss(h_pred, h_t)

def total_loss(seg, h_out, hbl, hvl, h_b, h_v, seg_t, h_t):
    # Segmentation
    L_seg = (focal_bce(seg[:,0:1], seg_t[:,0:1], gamma=2.0, w=8.0)
           + focal_bce(seg[:,1:2], seg_t[:,1:2], gamma=1.0, w=3.0)
           + focal_bce(seg[:,2:3], seg_t[:,2:3], gamma=1.0, w=3.0))
    L_iou = (2.0*soft_iou_loss(seg, seg_t, 0)
           + 1.5*soft_iou_loss(seg, seg_t, 1)
           + 0.5*soft_iou_loss(seg, seg_t, 2))
    # Height — dual branches supervised by their own class coverage
    L_hb = branch_height_loss(h_b, hbl, h_t, seg_t[:,0:1])
    L_hv = branch_height_loss(h_v, hvl, h_t, seg_t[:,1:2])
    # Combined fallback on all pixels
    w_c  = 1.0 + 2.0*((seg_t[:,0:1]>0.05)|(seg_t[:,1:2]>0.05)).float()
    L_hc = (w_c * F.huber_loss(h_out.float(), h_t.float(), delta=1.0, reduction='none')).mean()
    L_h  = 1.5*L_hb + 1.5*L_hv + 0.5*L_hc
    return L_seg + L_iou + 3.0*L_h

# ── Metrics ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def compute_miou(logits, target, thr=0.5):
    p = (torch.sigmoid(logits)>thr).float(); t = (target>thr).float()
    inter = (p*t).sum(dim=(0,2,3)); union = (p+t-p*t).sum(dim=(0,2,3))
    iou = inter/(union+1e-6)
    return iou.mean().item(), iou[0].item(), iou[1].item(), iou[2].item()

@torch.no_grad()
def masked_rmse(h_pred_log, h_t_log, mask, hmax=1000.0):
    if mask.sum() < 1: return float('nan')
    p = torch.expm1(h_pred_log).clamp(0,hmax)[mask]
    t = torch.expm1(h_t_log  ).clamp(0,hmax)[mask]
    return torch.sqrt(F.mse_loss(p,t)).item()

def proxy_score(ib,iv,iw,rb,rv):
    hb = 0.0 if (rb!=rb) else 1.0/(1.0+rb)
    hv = 0.0 if (rv!=rv) else 1.0/(1.0+rv)
    return 0.25*ib + 0.15*iv + 0.15*iw + 0.25*hb + 0.20*hv

@torch.no_grad()
def run_eval(m, loader):
    m.eval(); vl=0.0
    sp,st,hbp,hvp,ht = [],[],[],[],[]
    for x,tm,th,seg_t,h_t in loader:
        x   = x.to(DEVICE,non_blocking=True).to(memory_format=torch.channels_last)
        tm  = tm.to(DEVICE,non_blocking=True)
        th_ = th.to(DEVICE,non_blocking=True) if USE_THOR else None
        seg_t = seg_t.to(DEVICE,non_blocking=True)
        h_t   = h_t.to(DEVICE,non_blocking=True)
        with torch.amp.autocast('cuda'):
            sg,ho,hbl,hvl,hb,hv = m(x,tm,th_)
            vl += total_loss(sg,ho,hbl,hvl,hb,hv,seg_t,h_t).item()
        sp.append(sg.float().cpu()); st.append(seg_t.float().cpu())
        hbp.append(hb.float().cpu()); hvp.append(hv.float().cpu())
        ht.append(h_t.float().cpu())
    vl /= len(loader)
    sp,st   = torch.cat(sp), torch.cat(st)
    hbp,hvp,ht = torch.cat(hbp), torch.cat(hvp), torch.cat(ht)
    _,ib,iv,iw = compute_miou(sp, st)
    rb = masked_rmse(hbp, ht, st[:,0:1]>0.5)
    rv = masked_rmse(hvp, ht, st[:,1:2]>0.5)
    return dict(val_loss=vl, iou_b=ib, iou_v=iv, iou_w=iw,
                rmse_b=rb, rmse_v=rv, score=proxy_score(ib,iv,iw,rb,rv))

print('Loss & metrics ready.')


Loss & metrics ready.


## D2 — Diagnostic: prove the loss is finite *before* training
If v4 gave `trn=0.000 / val=nan`, every batch's loss was non-finite and skipped.
This runs **one** batch and localises any remaining NaN/Inf. Expect `TOTAL finite = True`.

In [ ]:
# ── v5 DIAGNOSTIC — one batch, localise any NaN/Inf before spending GPU time ──
model.eval()
xb, tmb, thb, segb, hgb = next(iter(trn_loader))
xb  = xb.to(DEVICE).to(memory_format=torch.channels_last)
tmb = tmb.to(DEVICE)
thb = thb.to(DEVICE) if USE_THOR else None
segb, hgb = segb.to(DEVICE), hgb.to(DEVICE)

def _rep(name, t):
    t = t.float()
    print(f'  {name:8s} shape={tuple(t.shape)}  finite={torch.isfinite(t).all().item()}  '
          f'range=[{t.min().item():+.2f}, {t.max().item():+.2f}]')

print('INPUTS (after dataset clamp — should be within ±%.0f):' % FEAT_CLAMP)
for n,t in [('spatial',xb),('tm',tmb),('seg',segb),('height',hgb)]: _rep(n,t)
if USE_THOR: _rep('thor', thb)

with torch.no_grad(), torch.amp.autocast('cuda'):
    sg, ho, hbl, hvl, h_b, h_v = model(xb, tmb, thb)
print('MODEL OUTPUTS:')
for n,t in [('seg',sg),('h_out',ho),('hbl',hbl),('hvl',hvl),('h_b',h_b),('h_v',h_v)]: _rep(n,t)

with torch.no_grad(), torch.amp.autocast('cuda'):
    L_seg = (focal_bce(sg[:,0:1],segb[:,0:1],2.0,8.0)
           + focal_bce(sg[:,1:2],segb[:,1:2],1.0,3.0)
           + focal_bce(sg[:,2:3],segb[:,2:3],1.0,3.0))
    L_iou = (2.0*soft_iou_loss(sg,segb,0)+1.5*soft_iou_loss(sg,segb,1)+0.5*soft_iou_loss(sg,segb,2))
    L_hb  = branch_height_loss(h_b, hbl, hgb, segb[:,0:1])
    L_hv  = branch_height_loss(h_v, hvl, hgb, segb[:,1:2])
    L_tot = total_loss(sg, ho, hbl, hvl, h_b, h_v, segb, hgb)
print('LOSS TERMS:')
for n,v in [('L_seg',L_seg),('L_iou',L_iou),('L_hbuild',L_hb),('L_hveg',L_hv),('TOTAL',L_tot)]:
    print(f'  {n:8s}= {v.item():+.4f}   finite={torch.isfinite(v).item()}')
print(f'\nTOTAL finite = {torch.isfinite(L_tot).item()}   '
      f'→ if True the NaN bug is gone and §G will train.')
model.train(); torch.cuda.empty_cache()


INPUTS (after dataset clamp — should be within ±15):
  spatial  shape=(16, 192, 256, 256)  finite=True  range=[-7.95, +7.38]
  tm       shape=(16, 768, 16, 16)  finite=True  range=[-15.00, +15.00]
  seg      shape=(16, 3, 256, 256)  finite=True  range=[+0.00, +1.00]
  height   shape=(16, 1, 256, 256)  finite=True  range=[+0.00, +3.89]
  thor     shape=(16, 768, 16, 16)  finite=True  range=[-15.00, +15.00]
MODEL OUTPUTS:
  seg      shape=(16, 3, 256, 256)  finite=True  range=[-0.11, +0.11]
  h_out    shape=(16, 1, 256, 256)  finite=True  range=[+2.75, +2.75]
  hbl      shape=(16, 64, 256, 256)  finite=True  range=[-0.14, +0.12]
  hvl      shape=(16, 64, 256, 256)  finite=True  range=[-0.13, +0.12]
  h_b      shape=(16, 1, 256, 256)  finite=True  range=[+2.75, +2.75]
  h_v      shape=(16, 1, 256, 256)  finite=True  range=[+2.75, +2.75]
LOSS TERMS:
  L_seg   = +3.3944   finite=True
  L_iou   = +3.4958   finite=True
  L_hbuild= +4.2931   finite=True
  L_hveg  = +7.2604   finite=True
  TOTA

## G — Training
Saves `final_best.pth` and `final_last.pth` to Drive after every epoch.
**If Colab disconnects: just Run all — it resumes from the last epoch automatically.**

In [ ]:
class EMA:
    def __init__(self, model, decay):
        self.decay  = decay
        self.shadow = {k: v.detach().clone() for k,v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k,v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
            else:
                self.shadow[k].copy_(v)
    def copy_to(self, model): model.load_state_dict(self.shadow, strict=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS,
    steps_per_epoch=len(trn_loader), pct_start=0.1)
scaler    = torch.amp.GradScaler('cuda')
ema       = EMA(model, EMA_DECAY)
ema_model = FusionNetV4(SPATIAL_CH, TM_CH, THOR_CH).to(DEVICE).to(memory_format=torch.channels_last)

start_epoch, best_score = 1, -1.0
history = json.loads(HISTORY_F.read_text()) if HISTORY_F.exists() else []

if CKPT_LAST.exists():
    ck = torch.load(CKPT_LAST, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler'])
    scaler.load_state_dict(ck['scaler'])
    ema.shadow = {k: v.to(DEVICE) for k,v in ck['ema'].items()}
    start_epoch = ck['epoch'] + 1
    best_score  = ck.get('best_score', -1.0)
    print(f'Resumed from epoch {ck["epoch"]}  best={best_score:.4f}')
else:
    print('Starting fresh.')
print(f'Epochs {start_epoch}→{EPOCHS}  hours left {hours_left():.1f}')

# ── quick data sanity check before wasting compute ────────────────────────────
_ae, _tm, _th, _seg, _h = trn_ds[0]
for _name, _t in [('ae',_ae),('tm',_tm),('th',_th),('seg',_seg),('h',_h)]:
    if _t.isnan().any() or _t.isinf().any():
        print(f'WARNING: {_name} contains NaN/Inf — check PREP normalisation')
    else:
        print(f'  data ok: {_name} {tuple(_t.shape)} [{_t.min():.2f}, {_t.max():.2f}]')

epoch_times = []
for epoch in range(start_epoch, EPOCHS+1):
    est = max(epoch_times[-3:] or [0]) / 3600 * 1.5
    if hours_left() < max(est, 0.15):
        print(f'⏰ Time budget at epoch {epoch} — Run all again to resume.'); break

    t0 = time.time(); model.train(); trn = 0.0; n_nan = 0
    for x,tm,th,seg_t,h_t in trn_loader:
        x     = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        tm    = tm.to(DEVICE, non_blocking=True)
        th_   = th.to(DEVICE, non_blocking=True) if USE_THOR else None
        seg_t = seg_t.to(DEVICE, non_blocking=True)
        h_t   = h_t.to(DEVICE,  non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            sg, ho, hbl, hvl, hb, hv = model(x, tm, th_)
            loss = total_loss(sg, ho, hbl, hvl, hb, hv, seg_t, h_t)
        if not torch.isfinite(loss):
            n_nan += 1; continue          # skip bad batch, don't update
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        prev_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        # OneCycleLR must step every batch AFTER optimizer — guard against
        # skipped steps (when scaler reduces scale due to inf gradients)
        if scaler.get_scale() >= prev_scale:
            scheduler.step()
        ema.update(model)
        trn += loss.item()
    n_batches = len(trn_loader) - n_nan
    trn = trn / max(n_batches, 1)
    if n_nan > 0:
        print(f'  WARNING: {n_nan}/{len(trn_loader)} NaN batches skipped this epoch')

    mv = run_eval(model, val_loader)
    is_best = mv['score'] > best_score
    if is_best:
        best_score = mv['score']
        torch.save({'epoch':epoch,'which':'raw','model':model.state_dict(),
                    'spatial_ch':SPATIAL_CH,'tm_ch':TM_CH,'thor_ch':THOR_CH,
                    'score':mv['score'],'metrics':mv}, CKPT_BEST)
    torch.save({'epoch':epoch,'model':model.state_dict(),
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'ema':ema.shadow,
                'best_score':best_score}, CKPT_LAST)
    history.append(dict(epoch=epoch, trn=round(trn,4), val=round(mv['val_loss'],4),
                        iou_b=round(mv['iou_b'],4), iou_v=round(mv['iou_v'],4),
                        iou_w=round(mv['iou_w'],4), rmse_b=round(mv['rmse_b'],3),
                        rmse_v=round(mv['rmse_v'],3), score=round(mv['score'],4)))
    HISTORY_F.write_text(json.dumps(history, indent=2))
    epoch_times.append(time.time()-t0)
    print(f'Ep {epoch:03d} [{epoch_times[-1]:4.0f}s|{hours_left():.1f}h] '
          f'trn={trn:.3f} val={mv["val_loss"]:.3f} | '
          f'IoU b={mv["iou_b"]:.3f} v={mv["iou_v"]:.3f} w={mv["iou_w"]:.3f} | '
          f'RMSE b={mv["rmse_b"]:.2f} v={mv["rmse_v"]:.2f} | '
          f'score={mv["score"]:.4f}{"  ✓BEST" if is_best else ""}')

# EMA final eval
ema.copy_to(ema_model)
me = run_eval(ema_model, val_loader)
print(f'\nEMA score: {me["score"]:.4f}  (best raw: {best_score:.4f})')
if me['score'] > best_score:
    best_score = me['score']
    torch.save({'epoch':'ema','which':'EMA','model':ema.shadow,
                'spatial_ch':SPATIAL_CH,'tm_ch':TM_CH,'thor_ch':THOR_CH,
                'score':me['score'],'metrics':me}, CKPT_BEST)
    print('EMA beat raw → saved as best.')
print(f'\nBest proxy score = {best_score:.4f}')


Resumed from epoch 20  best=0.4033
Epochs 21→80  hours left 10.7
  data ok: ae (192, 256, 256) [-6.83, 5.06]
  data ok: tm (768, 16, 16) [-5.11, 5.18]
  data ok: th (768, 16, 16) [-15.00, 15.00]
  data ok: seg (3, 256, 256) [0.00, 1.00]
  data ok: h (1, 256, 256) [0.00, 3.10]
Ep 021 [ 740s|10.5h] trn=24.585 val=24.016 | IoU b=0.318 v=0.843 w=0.668 | RMSE b=3.40 v=3.26 | score=0.4097  ✓BEST
Ep 022 [ 680s|10.3h] trn=24.994 val=24.123 | IoU b=0.316 v=0.835 w=0.652 | RMSE b=3.89 v=3.30 | score=0.3995
Ep 023 [ 696s|10.1h] trn=24.951 val=24.027 | IoU b=0.316 v=0.840 w=0.661 | RMSE b=4.08 v=3.29 | score=0.4001
Ep 024 [ 677s|9.9h] trn=24.791 val=24.054 | IoU b=0.326 v=0.841 w=0.667 | RMSE b=3.65 v=3.30 | score=0.4079
Ep 025 [ 695s|9.7h] trn=24.762 val=23.799 | IoU b=0.319 v=0.841 w=0.667 | RMSE b=3.92 v=3.28 | score=0.4034
Ep 026 [ 697s|9.5h] trn=24.684 val=23.828 | IoU b=0.322 v=0.842 w=0.664 | RMSE b=3.62 v=3.25 | score=0.4075
Ep 027 [ 690s|9.3h] trn=24.432 val=23.789 | IoU b=0.329 v=0.842 w

KeyboardInterrupt: 

## H — Training curves

In [ ]:
import matplotlib.pyplot as plt
h  = json.loads(HISTORY_F.read_text())
ep = [r['epoch'] for r in h]
fig, ax = plt.subplots(1,3,figsize=(16,4))
ax[0].plot(ep,[r['trn'] for r in h],label='train'); ax[0].plot(ep,[r['val'] for r in h],label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(ep,[r['iou_b'] for r in h],label='build')
ax[1].plot(ep,[r['iou_v'] for r in h],label='veg')
ax[1].plot(ep,[r['iou_w'] for r in h],label='water')
ax[1].set_title('IoU'); ax[1].legend()
ax[2].plot(ep,[r['score'] for r in h],'k-')
ax[2].axhline(0.5, color='r', ls='--', label='target 0.50')
ax[2].set_title('Proxy score'); ax[2].legend()
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Best proxy so far: {max(r["score"] for r in h):.4f}')


## I — Per-class threshold sweep

In [ ]:
ck   = torch.load(CKPT_BEST, map_location=DEVICE, weights_only=False)
best = FusionNetV4(ck['spatial_ch'], ck['tm_ch'], ck.get('thor_ch',0)).to(DEVICE).to(memory_format=torch.channels_last)
best.load_state_dict(ck['model']); best.eval()
print(f'Loaded: epoch={ck["epoch"]}  which={ck["which"]}  proxy={ck["score"]:.4f}')

logits_all, targets_all = [], []
with torch.no_grad():
    for x,tm,th,seg_t,_ in val_loader:
        x   = x.to(DEVICE).to(memory_format=torch.channels_last)
        tm_ = tm.to(DEVICE)
        th_ = th.to(DEVICE) if USE_THOR else None
        with torch.amp.autocast('cuda'):
            sg,_,_,_,_,_ = best(x, tm_, th_)
        logits_all.append(sg.float().cpu()); targets_all.append(seg_t.float())

probs = torch.sigmoid(torch.cat(logits_all))
t     = (torch.cat(targets_all) > 0.5).float()
BEST_THR = {}
for ch, name in enumerate(['build','veg','water']):
    bi, bt = 0.0, 0.5
    for thr in np.arange(0.10, 0.80, 0.02):
        p = (probs[:,ch] > thr).float()
        inter = (p*t[:,ch]).sum(); union = (p+t[:,ch]-p*t[:,ch]).sum()
        iou   = (inter/(union+1e-6)).item()
        if iou > bi: bi, bt = iou, round(float(thr), 2)
    BEST_THR[name] = bt
    print(f'  {name:6s}: best_thr={bt:.2f}  val_IoU={bi:.4f}')
THRESH_F.write_text(json.dumps(BEST_THR))
print('Thresholds saved.')


Loaded: epoch=27  which=raw  proxy=0.4153


RuntimeError: DataLoader worker (pid(s) 13876, 13877) exited unexpectedly

## J — Inference → submission.zip
Frees train zips first, stages test zips, runs 6-view TTA, writes submission.

In [ ]:
# Free big train zips to make room for test zips
for z in ['ae_train_npy.zip','ts_train_npy.zip','th_train_npy.zip']:
    if (TMP/z).exists(): (TMP/z).unlink()
u,fr = disk_used_gb(); print(f'After freeing train zips: {u:.0f} GB used / {fr:.0f} GB free')

TEST_ZIPS = (['ae_test_npy.zip','tm_test_npy.zip']
             + (['ts_test_npy.zip'] if USE_TESSERA else [])
             + (['th_test_npy.zip'] if USE_THOR    else []))
stage_local(DATA_TEST, TMP, TEST_ZIPS + ['test_names.json'])

ae_te = ZipNpyReader(TMP/'ae_test_npy.zip')
tm_te = ZipNpyReader(TMP/'tm_test_npy.zip')
ts_te = ZipNpyReader(TMP/'ts_test_npy.zip') if USE_TESSERA else None
th_te = ZipNpyReader(TMP/'th_test_npy.zip') if USE_THOR    else None
names = json.loads((TMP/'test_names.json').read_text())

test_ids = sorted(
    ae_te.ids() & tm_te.ids()
    & (ts_te.ids() if USE_TESSERA else ae_te.ids())
    & (th_te.ids() if USE_THOR    else ae_te.ids())
    & set(names))
print(f'Test patches: {len(test_ids)}')
assert len(test_ids) == 946, f'Expected 946, got {len(test_ids)}'

# Load best checkpoint
ck   = torch.load(CKPT_BEST, map_location=DEVICE, weights_only=False)
best = FusionNetV4(ck['spatial_ch'], ck['tm_ch'], ck.get('thor_ch',0)).to(DEVICE).to(memory_format=torch.channels_last)
best.load_state_dict(ck['model']); best.eval()
print(f'Inference model: epoch={ck["epoch"]} proxy={ck["score"]:.4f}')

BEST_THR = json.loads(THRESH_F.read_text()) if THRESH_F.exists() else {'build':0.5,'veg':0.5,'water':0.5}
print('Thresholds:', BEST_THR)

def remap(p, thr):
    lo = p * (0.5 / max(thr, 1e-6))
    hi = 0.5 + (p-thr) * (0.5 / max(1.0-thr, 1e-6))
    return np.where(p < thr, lo, hi).clip(0,1)

VIEWS = [
    (lambda z:z,                    lambda z:z),
    (lambda z:torch.flip(z,[-1]),   lambda z:torch.flip(z,[-1])),
    (lambda z:torch.flip(z,[-2]),   lambda z:torch.flip(z,[-2])),
    (lambda z:torch.flip(z,[-1,-2]),lambda z:torch.flip(z,[-1,-2])),
    (lambda z:torch.rot90(z,1,[-2,-1]), lambda z:torch.rot90(z,-1,[-2,-1])),
    (lambda z:torch.rot90(z,3,[-2,-1]), lambda z:torch.rot90(z,-3,[-2,-1])),
]

@torch.no_grad()
def predict_tta(x_t, tm_t, th_t=None):
    segs, hgts = [], []
    with torch.amp.autocast('cuda'):
        for fwd, inv in VIEWS:
            th_in = fwd(th_t).contiguous() if th_t is not None else None
            sg,ho,_,_,hb,hv = best(
                fwd(x_t).contiguous(memory_format=torch.channels_last),
                fwd(tm_t).contiguous(), th_in)
            # Route: build pixels → build branch, veg pixels → veg branch
            pb = torch.sigmoid(sg[:,0:1]); pv = torch.sigmoid(sg[:,1:2])
            h_routed = torch.where(pb >= pv, hb, hv)
            segs.append(inv(torch.sigmoid(sg)))
            hgts.append(inv(h_routed))
    return torch.stack(segs).mean(0), torch.stack(hgts).mean(0)

SUB_PRED = TMP/'predictions'
if SUB_PRED.exists(): shutil.rmtree(SUB_PRED)
SUB_PRED.mkdir()

t0 = time.time()
for i, pid in enumerate(test_ids):
    ae = ae_te.get(pid).astype(np.float32)
    parts = [ae]
    if USE_TESSERA: parts.append(ts_te.get(pid).astype(np.float32))
    x_np  = np.concatenate(parts, axis=0)
    tm_np = tm_te.get(pid).astype(np.float32)
    th_np = th_te.get(pid).astype(np.float32) if USE_THOR else None

    x_t  = torch.from_numpy(x_np)[None].to(DEVICE).nan_to_num_().clamp_(-FEAT_CLAMP, FEAT_CLAMP)
    tm_t = torch.from_numpy(tm_np)[None].to(DEVICE).nan_to_num_().clamp_(-FEAT_CLAMP, FEAT_CLAMP)
    th_t = (torch.from_numpy(th_np)[None].to(DEVICE).nan_to_num_().clamp_(-FEAT_CLAMP, FEAT_CLAMP)
            if th_np is not None else None)

    seg_p, h_p = predict_tta(x_t, tm_t, th_t)

    seg_np = seg_p[0].float().cpu().numpy()
    for ch, nm in enumerate(['build','veg','water']):
        seg_np[ch] = remap(seg_np[ch], BEST_THR[nm])

    h_np = torch.expm1(h_p).clamp(0,1000)[0].float().cpu().numpy()
    pred = np.concatenate([seg_np, h_np], axis=0).astype(np.float32)
    assert pred.shape == (4,256,256), pred.shape
    np.save(SUB_PRED/f'{names[pid]}.npy', pred)
    if (i+1) % 200 == 0:
        print(f'  {i+1}/946  ({(time.time()-t0)/(i+1):.2f}s/patch)')

with zipfile.ZipFile(SUB_ZIP,'w',zipfile.ZIP_DEFLATED) as zf:
    for npy in sorted(SUB_PRED.glob('*.npy')):
        zf.write(npy, f'predictions/{npy.name}')
sz = SUB_ZIP.stat().st_size/1e6
print(f'\nDONE → {SUB_ZIP}  ({sz:.0f} MB)  total {(time.time()-T_START)/3600:.2f} h')


## K — Sanity check

In [ ]:
with zipfile.ZipFile(SUB_ZIP) as zf:
    npys = [n for n in zf.namelist() if n.endswith('.npy')]
    print(f'NPY files in zip: {len(npys)}  (expect 946)')
    assert len(npys) == 946, f'Wrong count: {len(npys)}'
    assert all(n.startswith('predictions/') for n in npys)
    s = np.load(zf.open(sorted(npys)[0]))
print(f'Shape : {s.shape}  (expect (4,256,256))')
print(f'build : [{s[0].min():.3f}, {s[0].max():.3f}]')
print(f'veg   : [{s[1].min():.3f}, {s[1].max():.3f}]')
print(f'water : [{s[2].min():.3f}, {s[2].max():.3f}]')
print(f'height: [{s[3].min():.2f}, {s[3].max():.2f}] m')
assert s.shape == (4,256,256)
assert s[3].min() >= 0
print('\n✓ Submission looks good. Download from Drive:')
print(f'  {SUB_ZIP}')


## L — Multi-seed ensemble  *(optional, run after ≥2 seeds trained)*
Finds all `colab_work_v5_s*` folders, loads each best checkpoint,
averages predictions with multi-scale TTA, writes `submission_ensemble.zip`.

In [ ]:
import glob as _glob
MODEL_DIRS = sorted(_glob.glob(str(DRIVE_ROOT/'colab_work_v5_s*')))
assert MODEL_DIRS, 'No colab_work_v5_s* folders found — train at least one seed first.'

ens_models, ens_thr = [], []
for d in MODEL_DIRS:
    cp = Path(d)/'final_best.pth'
    if not cp.exists(): print(f'  skip (no ckpt): {Path(d).name}'); continue
    ck  = torch.load(cp, map_location=DEVICE, weights_only=False)
    mdl = FusionNetV4(ck['spatial_ch'], ck['tm_ch'], ck.get('thor_ch',0))
    mdl = mdl.to(DEVICE).to(memory_format=torch.channels_last)
    mdl.load_state_dict(ck['model']); mdl.eval(); ens_models.append(mdl)
    tf = Path(d)/'final_thresholds.json'
    ens_thr.append(json.loads(tf.read_text()) if tf.exists() else {'build':.5,'veg':.5,'water':.5})
    print(f'  {Path(d).name}: epoch={ck["epoch"]} proxy={ck["score"]:.4f}')
assert ens_models, 'No usable checkpoints.'
THR = {k: float(np.mean([t[k] for t in ens_thr])) for k in ['build','veg','water']}
print(f'Ensemble of {len(ens_models)} model(s) | thresholds: {THR}')

# Stage test zips (may already be present)
TEST_ZIPS = (['ae_test_npy.zip','tm_test_npy.zip']
             + (['ts_test_npy.zip'] if USE_TESSERA else [])
             + (['th_test_npy.zip'] if USE_THOR    else []))
stage_local(DATA_TEST, TMP, TEST_ZIPS + ['test_names.json'])
ae_te = ZipNpyReader(TMP/'ae_test_npy.zip'); tm_te = ZipNpyReader(TMP/'tm_test_npy.zip')
ts_te = ZipNpyReader(TMP/'ts_test_npy.zip') if USE_TESSERA else None
th_te = ZipNpyReader(TMP/'th_test_npy.zip') if USE_THOR    else None
names = json.loads((TMP/'test_names.json').read_text())
test_ids = sorted(
    ae_te.ids() & tm_te.ids()
    & (ts_te.ids() if USE_TESSERA else ae_te.ids())
    & (th_te.ids() if USE_THOR    else ae_te.ids())
    & set(names))
assert len(test_ids) == 946

SCALES = [0.75, 1.0, 1.25]
VIEWS  = [
    (lambda z:z,                    lambda z:z),
    (lambda z:torch.flip(z,[-1]),   lambda z:torch.flip(z,[-1])),
    (lambda z:torch.flip(z,[-2]),   lambda z:torch.flip(z,[-2])),
    (lambda z:torch.flip(z,[-1,-2]),lambda z:torch.flip(z,[-1,-2])),
    (lambda z:torch.rot90(z,1,[-2,-1]), lambda z:torch.rot90(z,-1,[-2,-1])),
    (lambda z:torch.rot90(z,3,[-2,-1]), lambda z:torch.rot90(z,-3,[-2,-1])),
]

def remap(p, thr):
    return np.where(p<thr, p*(0.5/max(thr,1e-6)),
                    0.5+(p-thr)*(0.5/max(1-thr,1e-6))).clip(0,1)

@torch.no_grad()
def ensemble_predict(x, tm, th=None):
    H,W = x.shape[-2:]; seg_s = hgt_s = None; n = 0
    with torch.amp.autocast('cuda'):
        for s in SCALES:
            xs = x if s==1.0 else F.interpolate(x, size=(int(round(H*s/16))*16,)*2,
                                                  mode='bilinear', align_corners=False)
            for fwd, inv in VIEWS:
                th_in = fwd(th).contiguous() if th is not None else None
                for mdl in ens_models:
                    sg,_,_,_,hb,hv = mdl(
                        fwd(xs).contiguous(memory_format=torch.channels_last),
                        fwd(tm).contiguous(), th_in)
                    pb = torch.sigmoid(sg[:,0:1]); pv = torch.sigmoid(sg[:,1:2])
                    hr = torch.where(pb >= pv, hb, hv)
                    sg = inv(torch.sigmoid(sg)); hr = inv(hr)
                    if sg.shape[-2:] != (H,W):
                        sg = F.interpolate(sg, (H,W), mode='bilinear', align_corners=False)
                        hr = F.interpolate(hr, (H,W), mode='bilinear', align_corners=False)
                    seg_s = sg if seg_s is None else seg_s+sg
                    hgt_s = hr if hgt_s is None else hgt_s+hr
                    n += 1
    return seg_s/n, hgt_s/n

ENS_SUB  = DRIVE_ROOT/'submission_ensemble_v5.zip'
SUB_PRED = TMP/'predictions'; shutil.rmtree(SUB_PRED, ignore_errors=True); SUB_PRED.mkdir()
t0 = time.time()
for i, pid in enumerate(test_ids):
    ae = ae_te.get(pid).astype(np.float32)
    parts = [ae]
    if USE_TESSERA: parts.append(ts_te.get(pid).astype(np.float32))
    x_np  = np.concatenate(parts, axis=0)
    tm_np = tm_te.get(pid).astype(np.float32)
    th_np = th_te.get(pid).astype(np.float32) if USE_THOR else None
    x_t  = torch.from_numpy(x_np)[None].to(DEVICE).nan_to_num_().clamp_(-FEAT_CLAMP, FEAT_CLAMP)
    tm_t = torch.from_numpy(tm_np)[None].to(DEVICE).nan_to_num_().clamp_(-FEAT_CLAMP, FEAT_CLAMP)
    th_t = (torch.from_numpy(th_np)[None].to(DEVICE).nan_to_num_().clamp_(-FEAT_CLAMP, FEAT_CLAMP)
            if th_np is not None else None)
    sp, hp = ensemble_predict(x_t, tm_t, th_t)
    seg = sp[0].float().cpu().numpy()
    for ch, nm in enumerate(['build','veg','water']): seg[ch] = remap(seg[ch], THR[nm])
    h = torch.expm1(hp).clamp(0,1000)[0].float().cpu().numpy()
    pred = np.concatenate([seg, h], axis=0).astype(np.float32)
    assert pred.shape == (4,256,256)
    np.save(SUB_PRED/f'{names[pid]}.npy', pred)
    if (i+1)%200==0: print(f'  {i+1}/946  ({(time.time()-t0)/(i+1):.2f}s/patch)')

with zipfile.ZipFile(ENS_SUB,'w',zipfile.ZIP_DEFLATED) as zf:
    for npy in sorted(SUB_PRED.glob('*.npy')): zf.write(npy, f'predictions/{npy.name}')
SUB_ZIP = ENS_SUB
print(f'ENSEMBLE DONE → {ENS_SUB} ({ENS_SUB.stat().st_size/1e6:.0f} MB)')
